# Solution — SQL Overview
Notebook ini menjalankan query inti dengan SQLite dan menjaga grain satu baris per order.

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(':memory:')
orders = pd.DataFrame([
    ('O-001', 'C-01', 'Jakarta', 800000, 'paid', '2026-09-01'),
    ('O-002', 'C-02', 'JKT', 120000, 'paid', '2026-09-02'),
    ('O-003', 'C-03', 'Bandung', 450000, 'pending', '2026-09-02'),
], columns=['order_id','customer_id','origin_city','order_value_idr','payment_status','order_created_at'])
shipments = pd.DataFrame([
    ('S-001', 'O-001', 'in_transit', '2026-09-01 08:00:00'),
    ('S-002', 'O-001', 'delivered', '2026-09-03 10:00:00'),
    ('S-003', 'O-002', 'created', '2026-09-02 10:00:00'),
], columns=['shipment_id','order_id','status','event_time'])
orders.to_sql('orders', connection, index=False)
shipments.to_sql('shipments', connection, index=False)

In [ ]:
latest_shipments = pd.read_sql_query('''
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY event_time DESC) AS rn
  FROM shipments
)
SELECT order_id, status, event_time
FROM ranked
WHERE rn = 1
''', connection)

backlog = pd.read_sql_query('''
WITH latest AS (
  SELECT order_id, status,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY event_time DESC) AS rn
  FROM shipments
)
SELECT o.origin_city, COUNT(*) AS backlog_orders, SUM(o.order_value_idr) AS backlog_value_idr
FROM orders AS o
JOIN latest AS s ON s.order_id = o.order_id AND s.rn = 1
WHERE o.payment_status = 'paid' AND s.status <> 'delivered'
GROUP BY o.origin_city
ORDER BY backlog_orders DESC
''', connection)
latest_shipments, backlog

## Takeaway
Ringkas tabel dengan grain lebih detail sebelum menggabungkannya ke tabel dengan grain lebih kasar.